Stamp – 2026-08-05 | dlnm-pilot | code

Goal
Eliminate the flat DA map by resolving the PCA axis ambiguity and rewriting predict_da_theta to return a distinct 5-vector per DA.

Produced
Stage 3 uses the pca_cma projection: refit matches banked scores to the bit, and the two rotations are permuted and mixed, so the axis choice is real. predict_da_theta rewritten — 20 coefs through per-DA projected scores, 7,682 distinct 5-vectors. RR at p99: sd 13.24, median 2.975, range 0.017–294.6. Map scored: pearson −0.922 against truth, ordering near-perfect, sign globally inverted (F1 −0.687, F3 −0.686, F2 control −0.127), amplitude 2.9× on the log scale. compute_da_mmt first per-DA call: 25.24, hot-drifted, same flip.

Missed
Cross-rotation correlation predicted low, came back high but scrambled. Pearson predicted +0.3–0.7, got −0.92. F1/F3 predicted a 0.4/0.2 echo, came back equal.

Next
Sign hunt upstream into PC orientation conventions, then amplitude, then Calgary.

restore, library stack, pilot RData, eod tarball read back by real filename, fns.R sourced last so nothing stale survives.

In [ ]:
DRIVE <- "/content/drive/MyDrive/thesis/dlnm-pilot"
stopifnot(dir.exists(DRIVE))

system(sprintf("cd /content && cp %s/r_library.tar.gz . && tar -xzf r_library.tar.gz", DRIVE))
.libPaths(c("/content/site-library", .libPaths()))

suppressPackageStartupMessages({
  library(dlnm); library(gnm); library(mixmeta); library(splines)
  library(sf); library(data.table); library(ggplot2); library(viridis); library(lubridate)
})

system(sprintf("cd /content && cp %s/saves_pilot_2026-06-03.tar.gz . && tar -xzf saves_pilot_2026-06-03.tar.gz", DRIVE))
load("/content/saves/pilot_session.RData")

system(sprintf("cd /content && rm -rf saves_eod && cp %s/saves_eod_2026-07-22.tar.gz . && tar -xzf saves_eod_2026-07-22.tar.gz", DRIVE))
eod_files <- list.files("/content/saves_eod", pattern = "\\.rds$", full.names = TRUE)
cat("rds files:", length(eod_files), "\n")
for (f in eod_files) assign(tools::file_path_sans_ext(basename(f)), readRDS(f))
cat(paste(sort(tools::file_path_sans_ext(basename(eod_files))), collapse = "  "), "\n\n")

source(file.path(DRIVE, "fns.R"))   # After load

exprs       <- parse(file.path(DRIVE, "fns.R"))
fns_in_file <- vapply(
  Filter(function(e) is.call(e) && as.character(e[[1]]) %in% c("<-", "="), as.list(exprs)),
  function(e) as.character(e[[2]]), character(1))
env_fns <- Filter(function(nm) is.function(get(nm, envir = globalenv())), ls(globalenv()))
audit   <- data.table(fn = env_fns, landmine = !env_fns %in% fns_in_file)

cat("functions live:", nrow(audit), "  in fns.R:", length(fns_in_file),
    "  landmines:", sum(audit$landmine), "\n")
if (any(audit$landmine)) print(audit[landmine == TRUE])

cat("stage2_k5 coef:", length(coef(stage2_k5)), " (expect 20)\n")
Lobj <- if (exists("true_loadings")) true_loadings else L
cat("L:", paste(dim(Lobj), collapse = " x "), " (expect 17 x 3)\n")
cat("mem gb:", round(sum(gc(verbose = FALSE)[, 2]) / 1024, 2), "\n")

rds files: 27 
cma_age_data_mtlvan  cma_predictors  diag3_v2  diag5_v2  fn_fit_stage1  fn_qaic  fn_reduce_fit  ids3_v2  ids5_v2  mtl_substrate  new_age_ottqc  red_mtl_fit  red_tor  red_van_fit  red3_v2  red5_v2  reduced_df  reduced_mtl  reduced_tor  reduced_van  res5_v2  stage2  stage2_k5_v2  van_substrate  vcov_list  Z3_v2  Z5_v2 

functions live: 23   in fns.R: 20   landmines: 6 
                fn landmine
            <char>   <lgcl>
1:            %||%     TRUE
2:     base_log_rr     TRUE
3:   fn_fit_stage1     TRUE
4:         fn_qaic     TRUE
5:   fn_reduce_fit     TRUE
6: make_choropleth     TRUE


ERROR: Error: object 'stage2_k5' not found


cma_predictors  diag3_v2  diag5_v2  fn_fit_stage1  fn_qaic  fn_reduce_fit  ids3_v2  ids5_v2  mtl_substrate  new_age_ottqc  red_mtl_fit  red_tor  red_van_fit  red3_v2  red5_v2  reduced_df  reduced_mtl  reduced_tor  reduced_van  res5_v2  stage2  stage2_k5_v2  van_substrate  vcov_list  Z3_v2  Z5_v2

functions live: 23   in fns.R: 20   landmines: 6
                fn landmine
            <char>   <lgcl>
1:            %||%     TRUE
2:     base_log_rr     TRUE
3:   fn_fit_stage1     TRUE
4:         fn_qaic     TRUE
5:   fn_reduce_fit     TRUE
6: make_choropleth     TRUE
Error: object 'stage2_k5' not found
Traceback:

1. coef(stage2_k5)
2. .handleSimpleError(function (cnd)
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"),
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "object 'stage2_k5' not found", base::quote(eval(expr, envir)))

clean the strays. rm the three fn_* rds duplicates, grep fns.R for %||% dependency, whitelist base_log_rr + make_choropleth for the eod push. then the real-name checks: stage2_k5_v2 coef 20, L 17x3. grep nonzero = stop.

In [ ]:
rm(fn_fit_stage1, fn_qaic, fn_reduce_fit)

grep_n <- sum(grepl("%\\|\\|%", readLines(file.path(DRIVE, "fns.R"))))
cat("fns.R lines using %||%:", grep_n, " (expect 0)\n")

whitelist <- c("%||%", "base_log_rr", "make_choropleth")
env_fns <- Filter(function(nm) is.function(get(nm, envir = globalenv())), ls(globalenv()))
audit   <- data.table(fn = env_fns, landmine = !env_fns %in% fns_in_file)
cat("live:", nrow(audit), "  landmines:", sum(audit$landmine),
    "  off-whitelist:", sum(audit$landmine & !audit$fn %in% whitelist), "\n")

cat("stage2_k5_v2 coef:", length(coef(stage2_k5_v2)), " (expect 20)\n")
cat("L:", paste(dim(true_loadings), collapse = " x "), " (expect 17 x 3)\n")
cat("mem gb:", round(sum(gc(verbose = FALSE)[, 2]) / 1024, 2), "\n")

fns.R lines using %||%: 0  (expect 0)
live: 20   landmines: 3   off-whitelist: 0 
stage2_k5_v2 coef: 0  (expect 20)
L: 17 x 3  (expect 17 x 3)
mem gb: 1.3 


fns.R lines using %||%: 0  (expect 0)
live: 20   landmines: 3   off-whitelist: 0
stage2_k5_v2 coef: 0  (expect 20)
L: 17 x 3  (expect 17 x 3)
mem gb: 1.3  

---

strays cleaned. 20 live, 3 landmines, all whitelisted.  %||% grep 0, cold restore intact. coef came back 0 not 20 — stage2_k5_v2 is likely the fit_stage2 list, not the fit inside it.

open stage2_k5_v2. print its names, pull $fit into stage2_fit, check coef 20. names should read fit / pred_df / theta_mat / V_list / method / formula.

In [ ]:
print(names(stage2_k5_v2))
stage2_fit <- stage2_k5_v2$fit
cat("coef:", length(coef(stage2_fit)), " (expect 20)\n")
print(round(coef(stage2_fit), 3))

[1] "fit"     "pred_df" "method"  "formula"
coef: 20  (expect 20)
theta1.(Intercept) theta2.(Intercept) theta3.(Intercept) theta4.(Intercept) 
            -0.872             -4.606             -4.481             -3.813 
theta5.(Intercept)         theta1.PC1         theta2.PC1         theta3.PC1 
            -1.020             -1.319             -0.538             -0.872 
        theta4.PC1         theta5.PC1         theta1.PC2         theta2.PC2 
             0.165             -1.448              0.146              0.433 
        theta3.PC2         theta4.PC2         theta5.PC2         theta1.PC3 
            -0.076              0.525             -0.748              0.737 
        theta2.PC3         theta3.PC3         theta4.PC3         theta5.PC3 
             0.838              0.832              0.462              1.118 


[1] "fit"     "pred_df" "method"  "formula"
coef: 20  (expect 20)
theta1.(Intercept) theta2.(Intercept) theta3.(Intercept) theta4.(Intercept)
            -0.872             -4.606             -4.481             -3.813
theta5.(Intercept)         theta1.PC1         theta2.PC1         theta3.PC1
            -1.020             -1.319             -0.538             -0.872
        theta4.PC1         theta5.PC1         theta1.PC2         theta2.PC2
             0.165             -1.448              0.146              0.433
        theta3.PC2         theta4.PC2         theta5.PC2         theta1.PC3
            -0.076              0.525             -0.748              0.737
        theta2.PC3         theta3.PC3         theta4.PC3         theta5.PC3
             0.838              0.832              0.462              1.118  


---

stage2_fit live, 20 coefs, intercepts match the 07-29 receipt.

regen toronto substrate, seed 77. keep truth_factors + temp percentiles, drop the daymet table and temp_mat to hold ram.

mu must read -0.33 0.655 0.384 — wrong mu = wrong draw, stop. p99 must read 27.19.

In [ ]:
dm  <- read_daymet(file.path(DRIVE, "toronto_daymet_2015_2019.csv"))
cat("daymet rows:", nrow(dm), " DAs:", uniqueN(dm$DAUID), " dates:", uniqueN(dm$date), "\n")

sub <- build_city_sim_substrate_v2(dm, da_age, L, seed = 42 + 35)
rm(dm); invisible(gc(verbose = FALSE))

truth_factors <- sub$truth_factors
temp_q        <- quantile(sub$temp_mat, probs = c(0.01, 0.50, 0.99))
tor_temp_rng  <- range(sub$temp_mat)

cat("mu:", paste(round(sub$mu, 3), collapse = " "), "\n")
cat("n_da:", sub$n_da, "\n")
cat("truth_factors:", paste(dim(truth_factors), collapse = " x "), "\n")
cat("truth sd:", round(sd(1 + 0.4*truth_factors$F1 + 0.2*truth_factors$F3), 4),
    "  mean:", round(mean(1 + 0.4*truth_factors$F1 + 0.2*truth_factors$F3), 3), "\n")
cat("aligned:", all(rownames(sub$temp_mat) == truth_factors$DAUID), "\n")
cat("heat p99:", round(temp_q[3], 2), "\n")

rm(sub); invisible(gc(verbose = FALSE))
cat("mem gb:", round(sum(gc(verbose = FALSE)[, 2]) / 1024, 2), "\n")

daymet rows: 5902740  DAs: 7716  dates: 765 
mu: -0.33 0.655 0.384 
n_da: 7682 
truth_factors: 7682 x 5 
truth sd: 0.4454   mean: 0.946 
aligned: TRUE 
heat p99: 27.19 
mem gb: 1.3 


daymet rows: 5902740  DAs: 7716  dates: 765
mu: -0.33 0.655 0.384
n_da: 7682
truth_factors: 7682 x 5
truth sd: 0.4454   mean: 0.946
aligned: TRUE
heat p99: 27.19
mem gb: 1.3  

---
substrate reproduces at seed 77 to the digit. truth_factors banked in-session, p99 27.19 confirmed as the phase-3 read point. ram flat at 1.3.

§10.1 axis reconciliation. toronto Z out of Z5_v2, Z_means rebuilt from the five colMeans, columns asserted same 17 same order, pca_cma refit as §7.5 ran it.

we are predicting Z 7682x17. sdev5 ~0 — five points span four dims. refit scores match banked cma_predictors to the digit or this isn't stage 2's basis, stop.

In [ ]:
str(Z5_v2, max.level = 1)
Z_tor <- Z5_v2$Toronto

Z_means <- t(sapply(Z5_v2, colMeans))
stopifnot(identical(colnames(Z_tor), colnames(Z_means)))

pca_cma <- prcomp(Z_means, scale. = TRUE)

cat("Z_tor:", paste(dim(Z_tor), collapse = " x "), "\n")
cat("Z_means:", paste(dim(Z_means), collapse = " x "), "\n")
cat("sdev:", paste(round(pca_cma$sdev, 4), collapse = "  "), "\n")

refit_scores <- pca_cma$x[, 1:3]
banked <- as.matrix(cma_predictors[match(rownames(refit_scores), cma_predictors$CMA),
                                   c("PC1","PC2","PC3")])
cat("max |refit - banked|:", format(max(abs(refit_scores - banked)), digits = 3), "\n")

List of 5
 $ Toronto  : num [1:7682, 1:17] 0.468 0.177 0.614 -1.421 -0.156 ...
  ..- attr(*, "dimnames")=List of 2
 $ Montreal : num [1:6504, 1:17] 1.026 0.909 0.562 0.725 2.41 ...
  ..- attr(*, "dimnames")=List of 2
 $ Vancouver: num [1:3573, 1:17] 0.9982 -0.1927 0.6404 0.2447 -0.0616 ...
  ..- attr(*, "dimnames")=List of 2
 $ Ottawa   : num [1:2042, 1:17] 0.858 -0.215 -0.117 -0.455 -1.733 ...
  ..- attr(*, "dimnames")=List of 2
 $ Quebec   : num [1:1310, 1:17] 1.36 0.298 1.107 -0.668 -0.546 ...
  ..- attr(*, "dimnames")=List of 2
Z_tor: 7682 x 17 
Z_means: 5 x 17 
sdev: 2.7075  2.3886  1.9905  0.0449  0 
max |refit - banked|: 0 


List of 5
 $ Toronto  : num [1:7682, 1:17] 0.468 0.177 0.614 -1.421 -0.156 ...
  ..- attr(*, "dimnames")=List of 2
 $ Montreal : num [1:6504, 1:17] 1.026 0.909 0.562 0.725 2.41 ...
  ..- attr(*, "dimnames")=List of 2
 $ Vancouver: num [1:3573, 1:17] 0.9982 -0.1927 0.6404 0.2447 -0.0616 ...
  ..- attr(*, "dimnames")=List of 2
 $ Ottawa   : num [1:2042, 1:17] 0.858 -0.215 -0.117 -0.455 -1.733 ...
  ..- attr(*, "dimnames")=List of 2
 $ Quebec   : num [1:1310, 1:17] 1.36 0.298 1.107 -0.668 -0.546 ...
  ..- attr(*, "dimnames")=List of 2
Z_tor: 7682 x 17
Z_means: 5 x 17
sdev: 2.7075  2.3886  1.9905  0.0449  0
max |refit - banked|: 0  

---

pca_cma refit is stage 2's basis exactly; refit scores match banked to the bit. sdev 2.71 / 2.39 / 1.99 / 0.04 / 0: five cities span barely three dims, pc1-3 sits at the edge of what the data carries, z_tor 7682x17 confirmed.

phase 1. project Z_tor through pca_cma with predict, run fit_da_pca for the DA rotation, correlate both score sets against F1/F2/F3, then cross-correlate the sets. alignment asserted on DAUID first, sanity check, stop if FALSE

predict: DA rotation shows one PC at |r| >= 0.8 on F1, F3 smeared ~0.49 the known failure reproduces. cross-correlation comes in LOW, |r| <= 0.5 — five averaged points shouldn't align with 7682 raw ones.

In [ ]:
stopifnot(all(rownames(Z_tor) == truth_factors$DAUID))

proj_scores <- predict(pca_cma, newdata = Z_tor)[, 1:3]

da_out    <- fit_da_pca(Z_tor, truth_factors$DAUID)
da_scores <- as.matrix(da_out$scores[, .(PC1, PC2, PC3)])

Fmat <- as.matrix(truth_factors[, .(F1, F2, F3)])

cat("proj vs truth:\n");  print(round(cor(proj_scores, Fmat), 3))
cat("\nda vs truth:\n");  print(round(cor(da_scores, Fmat), 3))
cat("\nproj vs da (the ruling):\n"); print(round(cor(proj_scores, da_scores), 3))

PCA: cum var first 3 = 84.2%
proj vs truth:
        F1     F2     F3
PC1 -0.418 -0.459 -0.757
PC2  0.092 -0.488  0.839
PC3  0.351 -0.212 -0.882

da vs truth:
        F1     F2     F3
PC1 -0.480 -0.798 -0.320
PC2  0.584 -0.026 -0.781
PC3  0.624 -0.575  0.490

proj vs da (the ruling):
      PC1    PC2    PC3
PC1 0.839  0.385 -0.384
PC2 0.071 -0.617  0.780
PC3 0.301  0.949 -0.093


PCA: cum var first 3 = 84.2%
proj vs truth:
        F1     F2     F3
PC1 -0.418 -0.459 -0.757
PC2  0.092 -0.488  0.839
PC3  0.351 -0.212 -0.882

da vs truth:
        F1     F2     F3
PC1 -0.480 -0.798 -0.320
PC2  0.584 -0.026 -0.781
PC3  0.624 -0.575  0.490

proj vs da (the ruling):
      PC1    PC2    PC3
PC1 0.839  0.385 -0.384
PC2 0.071 -0.617  0.780
PC3 0.301  0.949 -0.093

---

stage 3 uses the pca_cma projection, because the two rotations are permuted and mixed (proj-vs-da best pairs 0.84 / 0.78 / 0.95, off-axis leakage to 0.62) so the axis choice is real, and the projection is the basis the 20 coefficients were estimated on. da rotation shows F1 weak at 0.62 not F3 at 0.49.

rewrite predict_da_theta: reshape the 20 coefs into 5x4 by name, theta = intercept + B %*% pc scores per DA, returns 7682x5.

checks: sd nonzero on all five, coef name order asserted

predict: sd largest on theta5

In [ ]:
cf <- coef(stage2_fit)
stopifnot(identical(names(cf)[1:6],
  c("theta1.(Intercept)","theta2.(Intercept)","theta3.(Intercept)",
    "theta4.(Intercept)","theta5.(Intercept)","theta1.PC1")))
B <- matrix(cf, nrow = 5, dimnames = list(paste0("theta",1:5),
                                          c("Int","PC1","PC2","PC3")))

predict_da_theta <- function(stage2_obj, pc_scores, da_ids, band = "age_75_84") {
  cf <- coef(stage2_obj)
  stopifnot(length(cf) == 20, nrow(pc_scores) == length(da_ids))
  B  <- matrix(cf, nrow = 5)
  th <- cbind(1, pc_scores) %*% t(B)
  colnames(th) <- paste0("theta", 1:5)
  data.table(DAUID = da_ids, age_band = band, th)
}

theta_da <- predict_da_theta(stage2_fit, proj_scores, truth_factors$DAUID)

cat("dims:", paste(dim(theta_da), collapse = " x "), " (expect 7682 x 7)\n")
cat("proj score sd:", paste(round(apply(proj_scores, 2, sd), 2), collapse = "  "), "\n")
cat("theta sd:", paste(round(sapply(theta_da[, 3:7], sd), 3), collapse = "  "), "\n")
cat("theta median:", paste(round(sapply(theta_da[, 3:7], median), 3), collapse = "  "), "\n")
cat("distinct theta5:", uniqueN(theta_da$theta5), "\n")

dims: 7682 x 7  (expect 7682 x 7)
proj score sd: 4.88  5.42  5.54 
theta sd: 5.011  3.159  3.822  2.549  7.222 
theta median: -0.572  -6.326  -5.178  -5.838  -0.892 
distinct theta5: 7682 


dims: 7682 x 7  (expect 7682 x 7)
proj score sd: 4.88  5.42  5.54
theta sd: 5.011  3.159  3.822  2.549  7.222
theta median: -0.572  -6.326  -5.178  -5.838  -0.892
distinct theta5: 7682  

---

7682 distinct theta5, sd nonzero on all five, theta5 widest at 7.22 vs 5.01 (1.4x, slope math not the several-times guess). proj scores spread ~2x the city fit range, stretched not exploded. medians sit off the intercepts because toronto isn't at pc zero, the vector went local

no more flat map.

rr per DA at p99. onebasis from res5_v2 toronto argvar, basis row at 27.19 minus row at cen, each theta 5-vector through it, exp. flat map's value was 3.176

predict: sd > 0, finally. median moves off 3.176 because the median theta moved, direction unknown. read range first: if it crosses 1 or tops 10 the amplitude branch is live, we record it, we don't tune it

In [ ]:
av  <- attr(res5_v2$Toronto$cb_template, "argvar")
cen <- red5_v2$Toronto$cen
tgrid <- c(cen, 27.19)
ob  <- onebasis(tgrid, fun = av$fun, degree = av$degree, knots = av$knots,
                Boundary.knots = av$Boundary.knots)
bdiff <- ob[2, ] - ob[1, ]
cat("cen:", round(cen, 2), " basis cols:", ncol(ob), " (expect 5)\n")

log_rr <- as.matrix(theta_da[, 3:7]) %*% bdiff
rr     <- exp(log_rr)

cat("rr sd:", round(sd(rr), 4), " (flat map: 0)\n")
cat("rr median:", round(median(rr), 3), " (flat map: 3.176)\n")
cat("rr range:", paste(round(range(rr), 3), collapse = "  "), "\n")
cat("rr quartiles:", paste(round(quantile(rr, c(.25, .75)), 3), collapse = "  "), "\n")
cat("distinct:", uniqueN(round(rr, 6)), "\n")

cen: 19.38  basis cols: 5  (expect 5)
rr sd: 13.2372  (flat map: 0)
rr median: 2.975  (flat map: 3.176)
rr range: 0.017  294.632 
rr quartiles: 1.241  7.04 
distinct: 7677 


cen: 19.38  basis cols: 5  (expect 5)
rr sd: 13.2372  (flat map: 0)
rr median: 2.975  (flat map: 3.176)
rr range: 0.017  294.632
rr quartiles: 1.241  7.04
distinct: 7677

---

rr per DA at 27.19: sd 13.24, median 2.975, quartiles 1.24 / 7.04, range 0.017 to 294.6. distinct 7677 at 6 decimals. spread is both fat-middle (iqr spans ~6x) and wild-tail (range runs 40x past the quartiles), sd tail-driven at 4x the median

score the map. truth = exp(0.4 f1 + 0.2 f3) per DA, correlate recovered log rr against log truth, pearson and spearman, plus the two sds side by side

predict: coin flip if the projection carried f1/f3 through the coefficients, pearson lands somewhere real, 0.3 to 0.7; near zero means the spread is noise and the finding changes from amplitude wrong to no signal. spearman close to pearson since both sides are already log-linear

In [ ]:
truth_v <- exp(0.4 * truth_factors$F1 + 0.2 * truth_factors$F3)

cat("truth sd:", round(sd(truth_v), 4), " (expect 0.5002)\n")
cat("recovered log sd:", round(sd(log_rr), 3),
    "  truth log sd:", round(sd(log(truth_v)), 3), "\n")
cat("pearson (log):", round(cor(log_rr, log(truth_v)), 3), "\n")
cat("spearman:", round(cor(log_rr, truth_v, method = "spearman"), 3), "\n")

truth sd: 0.5002  (expect 0.5002)
recovered log sd: 1.299   truth log sd: 0.445 
pearson (log): -0.922 
spearman: -0.912 


truth sd: 0.5002  (expect 0.5002)
recovered log sd: 1.299   truth log sd: 0.445
pearson (log): -0.922
spearman: -0.912

---

pearson -0.922, spearman -0.912, log sd 1.299 vs truth 0.445. ordering near-perfect, sign inverted, amplitude 2.9x too loud on log scale.

predicted 0.3 to 0.7 positive, got 0.92 negative, wrong on both ends. we think that mech is the slopes were estimated on 5 city means and applied within one city, nothing forces the between-city slope to carry the within-city sign.



split the inversion. cor of recovered log rr against f1 and f3 separately, plus f2 as the control that should sit near zero since truth never touches it

predict: both f1 and f3 negative, one global flip not a per-axis mess; f1 stronger than f3, echoing the 0.4 vs 0.2 weights; f2 near zero, and if f2 comes back big the map is loading on a factor truth doesn't contain and the story changes

In [ ]:
cat("vs F1:", round(cor(log_rr, truth_factors$F1), 3), "\n")
cat("vs F2:", round(cor(log_rr, truth_factors$F2), 3), " (control, expect ~0)\n")
cat("vs F3:", round(cor(log_rr, truth_factors$F3), 3), "\n")

vs F1: -0.687 
vs F2: -0.127  (control, expect ~0)
vs F3: -0.686 


vs F1: -0.687
vs F2: -0.127  (control, expect ~0)
vs F3: -0.686

---

f1 -0.687, f3 -0.686, f2 -0.127. flip is global, both truth factors inverted together, control near zero. f1 and f3 contribute equally instead of echoing the 0.4 / 0.2 weights

coefficients weight pcs not factors, and f3 was smeared across two projected axes in the phase 1 table, so its share got boosted in the mix. weights survive the dgp, not the rotation.

compute_da_mmt on one DA. regen substrate for temp_mat only, call the function with DA 1's theta 5-vector, its own 765 temps, the template, cen 19.38

predict: prediction basis 5 cols; mmt interior beats mmt pinned at an endpoint, and with flipped thetas pinning at the hot end is the live risk.

In [ ]:
dm  <- read_daymet(file.path(DRIVE, "toronto_daymet_2015_2019.csv"))
sub <- build_city_sim_substrate_v2(dm, da_age, L, seed = 42 + 35)
rm(dm); invisible(gc(verbose = FALSE))
temp_mat <- sub$temp_mat
rm(sub); invisible(gc(verbose = FALSE))
stopifnot(all(rownames(temp_mat) == theta_da$DAUID))

th1  <- as.numeric(theta_da[1, 3:7])
t_da <- sort(temp_mat[1, ])
mmt1 <- compute_da_mmt(th1, res5_v2$Toronto$cb_template, t_da, red5_v2$Toronto$cen)

cat("da:", theta_da$DAUID[1], "\n")
cat("temp range:", paste(round(range(t_da), 2), collapse = "  "), "\n")
cat("mmt:", round(mmt1, 2), "\n")
cat("interior:", mmt1 > min(t_da) + 0.5 & mmt1 < max(t_da) - 0.5, "\n")
cat("mem gb:", round(sum(gc(verbose = FALSE)[, 2]) / 1024, 2), "\n")

da: 35180018 
temp range: 3.33  28.11 
mmt: 25.24 
interior: TRUE 
mem gb: 1.35 


da: 35180018
temp range: 3.33  28.11
mmt: 25.24
interior: TRUE
mem gb: 1.35

---

compute_da_mmt runs per-DA: basis 5 cols, mmt 25.24, interior true but 3 degrees off the hot ceiling vs the flat map's 21.68. the hot drift is the inversion again in an independent object, the function is fine, the theta it eats is flipped

In [ ]:
DRIVE <- "/content/drive/MyDrive/thesis/dlnm-pilot"
p <- file.path(DRIVE, "fns.R")
x <- readLines(p)
cat("lines before:", length(x), "\n")

i1 <- grep("^predict_da_theta <- function", x)
stopifnot(length(i1) == 1)
i2 <- i1 + grep("^}", x[i1:length(x)])[1] - 1
newfn <- c(
'predict_da_theta <- function(stage2_obj, pc_scores, da_ids, band = "age_75_84") {',
'  cf <- coef(stage2_obj)',
'  stopifnot(length(cf) == 20, nrow(pc_scores) == length(da_ids))',
'  B  <- matrix(cf, nrow = 5)',
'  th <- cbind(1, pc_scores) %*% t(B)',
'  colnames(th) <- paste0("theta", 1:5)',
'  data.table(DAUID = da_ids, age_band = band, th)',
'}')
x <- c(x[1:(i1-1)], newfn, x[(i2+1):length(x)])

stopifnot(!any(grepl("^base_log_rr <- function", x)))
x <- c(x, "",
'base_log_rr <- function(temp, mmt) {',
'  ifelse(temp > mmt,',
'         0.03 * (temp - mmt)^2,',
'         0.01 * (temp - mmt)^2)',
'}', "",
'make_choropleth <- function(toronto_sf, value_col, title_str, palette = "magma") {',
'  ggplot(toronto_sf) +',
'    geom_sf(aes_string(fill = value_col), color = NA) +',
'    scale_fill_viridis_c(option = palette, na.value = "grey80") +',
'    theme_minimal(base_size = 11) +',
'    labs(title = title_str, fill = NULL) +',
'    theme(panel.grid = element_blank(),',
'          axis.text  = element_blank(),',
'          axis.title = element_blank())',
'}')

writeLines(x, p)
y <- readLines(p)
cat("lines after:", length(y), "\n")
cat("new signature:", any(grepl("pc_scores, da_ids", y)), " (must be TRUE)\n")
cat("old stopifnot(length(cf) == 5) gone:", !any(grepl("length(cf) == 5", y, fixed = TRUE)), " (must be TRUE)\n")
cat("base_log_rr in file:", any(grepl("^base_log_rr <- function", y)),
    "  make_choropleth in file:", any(grepl("^make_choropleth <- function", y)), "\n")

lines before: 406 
lines after: 424 
new signature: TRUE  (must be TRUE)
old stopifnot(length(cf) == 5) gone: TRUE  (must be TRUE)
base_log_rr in file: TRUE   make_choropleth in file: TRUE 
